# Working with Datastores - Azure ML SDK v2 Version

In previous labs, you connected to an Azure ML workspace and ran a simple experiment based on data in a local CSV file. Now you will work with data stored in Azure Machine Learning.

> **Updated version**: This notebook has been converted from the older Azure ML SDK v1 `SKLearn Estimator`, `DataReference`, and `Run` pattern to the newer SDK v2 `MLClient`, `command` job, `Input`, MLflow logging, and SDK v2 model registration pattern.

## Connect to Your Workspace

The first thing you need to do is connect to your workspace using the Azure ML SDK v2.

> **Note**: If your authenticated Azure session has expired, you may be prompted to authenticate again.


In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Load the workspace from config.json
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

workspace = ml_client.workspaces.get(ml_client.workspace_name)

print(f"Ready to use Azure ML workspace: {workspace.name}")
print(f"Resource group: {ml_client.resource_group_name}")
print(f"Subscription: {ml_client.subscription_id}")


## View Datastores in the Workspace

The workspace contains several datastores, including the **aml_data** datastore created in the previous task.

Run the following code to retrieve the default datastore and list all datastores in the workspace.


In [ ]:
# Get the default datastore
default_ds = ml_client.datastores.get_default()

print("Default datastore:", default_ds.name)
print()

# Enumerate all datastores, indicating which is the default
for ds in ml_client.datastores.list():
    print(ds.name, "- Default =", ds.name == default_ds.name)


## Get a Datastore to Work With

You want to work with the **aml_data** datastore, so you need to get it by name:

In [ ]:
# Get the datastore named aml_data
aml_datastore = ml_client.datastores.get("aml_data")

print("Datastore name:", aml_datastore.name)
print("Datastore type:", aml_datastore.type)


## Choose the Datastore / Data Location

In SDK v1, the lab used `ws.set_default_datastore()` and `DataReference`.

In SDK v2, the cleaner pattern is:

1. Upload local data as an Azure ML **Data asset**
2. Pass that Data asset into a `command` job as an `Input`
3. Let Azure ML mount or download the input folder for the training script

We will still use the workspace datastore behind the scenes, but the job will consume the data through an SDK v2 input.


In [ ]:
# We will use this datastore/data name in later cells
datastore_name = "aml_data"
data_asset_name = "diabetes-data-sdkv2"

print("Using datastore:", datastore_name)
print("Data asset name:", data_asset_name)


## Upload Data as an SDK v2 Data Asset

The old SDK v1 lab used:

```python
default_ds.upload_files(...)
```

In SDK v2, create a `Data` asset from the local `./data` folder. Azure ML uploads the folder and makes it reusable by jobs.


In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from datetime import datetime
import os

# Confirm local data files exist
print("Local data files:")
print(os.listdir("./data"))

# Use a timestamp version so the cell can be rerun without version conflicts
data_version = datetime.now().strftime("%Y%m%d%H%M%S")

diabetes_data = Data(
    name=data_asset_name,
    version=data_version,
    type=AssetTypes.URI_FOLDER,
    path="./data",
    description="Diabetes CSV files uploaded from local ./data folder"
)

diabetes_data = ml_client.data.create_or_update(diabetes_data)

print("Data asset created:")
print("Name:", diabetes_data.name)
print("Version:", diabetes_data.version)
print("Path:", diabetes_data.path)


## Train a Model from the Data Asset

In the old SDK v1 notebook, the code created a `DataReference` like this:

```python
data_ref = default_ds.path('diabetes-data').as_download(...)
```

In SDK v2, use an `Input` that points to the uploaded Data asset:

```python
Input(type="uri_folder", path="azureml:<data-name>:<version>")
```

Azure ML will make this folder available to the training script at runtime.


In [ ]:
from azure.ai.ml import Input

data_input = Input(
    type=AssetTypes.URI_FOLDER,
    path=f"azureml:{diabetes_data.name}:{diabetes_data.version}"
)

print("Training input:")
print(data_input)


To use the data input in a training script, define a parameter for it.

The next two code cells create:

1. A folder named **diabetes_training_from_datastore**
2. A script that trains a classification model using all CSV files in the input data folder


In [ ]:
import os

# Create a folder for the experiment files
experiment_folder = 'diabetes_training_from_datastore'
os.makedirs(experiment_folder, exist_ok=True)
print(experiment_folder, 'folder created.')

In [ ]:
%%writefile $experiment_folder/diabetes_training.py
# Import libraries
import os
import argparse
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Get parameters
parser = argparse.ArgumentParser()
parser.add_argument('--regularization', type=float, dest='reg_rate', default=0.01, help='regularization rate')
parser.add_argument('--data-folder', type=str, dest='data_folder', help='data folder path')
args = parser.parse_args()

reg = args.reg_rate
data_folder = args.data_folder

print("Loading data from", data_folder)
print("Files found:", os.listdir(data_folder))

# Load all CSV files and concatenate them into one dataframe
all_files = [file for file in os.listdir(data_folder) if file.endswith(".csv")]
diabetes = pd.concat(
    (pd.read_csv(os.path.join(data_folder, csv_file)) for csv_file in all_files),
    ignore_index=True
)

# Separate features and labels
X = diabetes[
    [
        'Pregnancies',
        'PlasmaGlucose',
        'DiastolicBloodPressure',
        'TricepsThickness',
        'SerumInsulin',
        'BMI',
        'DiabetesPedigree',
        'Age'
    ]
].values

y = diabetes['Diabetic'].values

# Split data into training set and test set
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=0
)

# Train a logistic regression model
print('Training a logistic regression model with regularization rate of', reg)

model = LogisticRegression(
    C=1/reg,
    solver="liblinear"
).fit(X_train, y_train)

# Calculate accuracy
y_hat = model.predict(X_test)
acc = np.average(y_hat == y_test)
print('Accuracy:', acc)

# Calculate AUC
y_scores = model.predict_proba(X_test)
auc = roc_auc_score(y_test, y_scores[:, 1])
print('AUC:', auc)

# Log metrics with MLflow - SDK v2 recommended pattern
mlflow.log_metric("Regularization Rate", float(reg))
mlflow.log_metric("Accuracy", float(acc))
mlflow.log_metric("AUC", float(auc))

# Save the model into outputs.
# Azure ML automatically captures files saved in the outputs folder.
os.makedirs('outputs', exist_ok=True)
joblib.dump(value=model, filename='outputs/diabetes_model.pkl')

# Also log the sklearn model as an MLflow artifact
mlflow.sklearn.log_model(model, artifact_path="model")

print("Model saved to outputs/diabetes_model.pkl")


The script will load the training data from the SDK v2 input path passed to it as a parameter.

Now create and submit an SDK v2 `command` job. This replaces the old SDK v1 `SKLearn` Estimator pattern.


In [ ]:
from azure.ai.ml import command
from azure.ai.ml.entities import Environment

# Create / reuse custom sklearn environment
sklearn_env = Environment(
    name="diabetes-sklearn-env",
    description="Environment for diabetes sklearn training",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file={
        "channels": ["conda-forge"],
        "dependencies": [
            "python=3.10",
            "pip",
            {
                "pip": [
                    "azureml-defaults",
                    "pandas",
                    "numpy",
                    "scikit-learn",
                    "joblib",
                    "mlflow",
                    "azureml-mlflow"
                ]
            }
        ]
    }
)

ml_client.environments.create_or_update(sklearn_env)

experiment_name = "diabetes-training"

job = command(
    code=experiment_folder,
    command="python diabetes_training.py --regularization 0.1 --data-folder ${{inputs.data_folder}}",
    inputs={
        "data_folder": data_input
    },
    environment="diabetes-sklearn-env@latest",
    compute="local",
    experiment_name=experiment_name,
    display_name="diabetes-training-sdkv2-local"
)

returned_job = ml_client.jobs.create_or_update(job)

print("Job submitted successfully.")
print("Job name:", returned_job.name)
print("Studio URL:", returned_job.studio_url)


The first time the experiment is run, it may take time to build the custom Python environment. Subsequent runs are usually faster.

When the job has completed, open the **Studio URL** above and review:

```text
Jobs -> your job -> Outputs + logs
```

You can also check the status from Python.


In [ ]:
# Refresh job status
job_details = ml_client.jobs.get(returned_job.name)

print("Job status:", job_details.status)
print("Job name:", job_details.name)
print("Studio URL:", job_details.studio_url)

# Download outputs and logs locally
ml_client.jobs.download(
    name=returned_job.name,
    download_path="./job_download",
    all=True
)

print("Downloaded job outputs and logs to ./job_download")
print()

# List downloaded files
for root, dirs, files in os.walk("./job_download"):
    for file in files:
        print(os.path.join(root, file))


## View Metrics and Register the Model

The old SDK v1 notebook used:

```python
run.get_metrics()
run.get_file_names()
run.register_model(...)
```

In SDK v2, the training script logs metrics with MLflow. Then the model can be registered with `ml_client.models.create_or_update()`.


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from azure.ai.ml.entities import Model

# Connect MLflow to this Azure ML workspace
tracking_uri = workspace.mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)

mlflow_client = MlflowClient()

# In Azure ML SDK v2, the job name is commonly the MLflow run id.
# This try/fallback makes the cell more robust.
try:
    mlflow_run = mlflow_client.get_run(returned_job.name)
except Exception:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=5
    )
    print(runs[["run_id", "status", "start_time"]])
    mlflow_run = mlflow_client.get_run(runs.iloc[0]["run_id"])

metrics = mlflow_run.data.metrics

print("Logged metrics:")
for key, value in metrics.items():
    print(key, value)

# Register model from the job's outputs folder
model_path = f"azureml://jobs/{returned_job.name}/outputs/artifacts/paths/outputs/diabetes_model.pkl"

registered_model = Model(
    path=model_path,
    name="diabetes_model",
    type=AssetTypes.CUSTOM_MODEL,
    description="Diabetes classification model trained with SDK v2 command job",
    tags={
        "Training context": "SDK v2 command job using Data asset"
    },
    properties={
        "AUC": str(metrics.get("AUC")),
        "Accuracy": str(metrics.get("Accuracy"))
    }
)

registered_model = ml_client.models.create_or_update(registered_model)

print()
print("Registered model:")
print("Name:", registered_model.name)
print("Version:", registered_model.version)


In this exercise, you explored how to work with data in Azure Machine Learning using the SDK v2 pattern:

1. Connect to the workspace with `MLClient`
2. View datastores
3. Upload local CSV files as a reusable Data asset
4. Pass the Data asset into a `command` training job
5. Log metrics with MLflow
6. Save the trained model to the job outputs
7. Register the model with SDK v2

This replaces the older SDK v1 `DataReference`, `SKLearn Estimator`, and `Run` API workflow.
